### Supplement 2: Structured outputs (`2-structured.py`)

Get the model's answer as data in a shape you define, instead of free text.

**The key point:** You describe the shape as a Pydantic class. `parse()` sends it to OpenAI as a JSON Schema, the model is forced to write JSON that matches it, and the SDK turns that JSON back into an instance of your class at `message.parsed`.

```
class CalendarEvent(BaseModel)       written by you
   │  parse(..., response_format=CalendarEvent)
   │  the SDK turns the class into a JSON Schema and sends it
   ▼
the model writes JSON that matches the schema
   │  the SDK turns that JSON into a CalendarEvent
   ▼
message.content    the JSON text                (str)
message.parsed     the same data as an object   (CalendarEvent)
```

Deep dive: [Exhaustive_2-structured.ipynb](Exhaustive_2-structured.ipynb).

#### 1. Setup
Same as Supplement 1, plus `BaseModel` from Pydantic.

In [1]:
import os

from openai import OpenAI
from pydantic import BaseModel
from dotenv import load_dotenv

import json  # added for the prints below

load_dotenv()

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

#### 2. The shape: `CalendarEvent`
A Pydantic class: each line declares a field and its type. The model never sees this Python code. The SDK calls the class's `model_json_schema()` (a Pydantic method, printed below) to get a JSON Schema, tightens it for strict mode, and sends that. Here, tightening only adds `"additionalProperties": false`, which forbids extra keys. `list[str]` becomes `"type": "array"` with string `items`.

In [2]:
class CalendarEvent(BaseModel):
    name: str
    date: str
    participants: list[str]

In [3]:
print(json.dumps(CalendarEvent.model_json_schema(), indent=2))

{
  "properties": {
    "name": {
      "title": "Name",
      "type": "string"
    },
    "date": {
      "title": "Date",
      "type": "string"
    },
    "participants": {
      "items": {
        "type": "string"
      },
      "title": "Participants",
      "type": "array"
    }
  },
  "required": [
    "name",
    "date",
    "participants"
  ],
  "title": "CalendarEvent",
  "type": "object"
}


#### 3. The API call: `parse()`
`parse()` is `create()` plus two jobs the SDK does for you: it puts the schema for `response_format=CalendarEvent` into the request, and it turns the model's JSON reply back into a `CalendarEvent`. The type below prints as `ParsedChatCompletion[TypeVar]`; that label is cosmetic, and step 4 shows the parsed value really is a `CalendarEvent`.

In [4]:
completion = client.chat.completions.parse(
    model="gpt-5-nano",
    messages=[
        {"role": "system", "content": "Extract the event information."},
        {
            "role": "user",
            "content": "Alice and Bob are going to a science fair on Friday.",
        },
    ],
    response_format=CalendarEvent,
)

In [5]:
print(type(completion))
# warnings=False hides a harmless Pydantic warning about the `parsed` field; it is still printed.
print(completion.model_dump_json(indent=2, warnings=False))

<class 'openai.types.chat.parsed_chat_completion.ParsedChatCompletion[TypeVar]'>
{
  "id": "chatcmpl-EPu2f8yBvaIVdkbRCTzZZDibdibNm",
  "choices": [
    {
      "finish_reason": "stop",
      "index": 0,
      "logprobs": null,
      "message": {
        "content": "{\"name\":\"Science Fair\",\"date\":\"Friday\",\"participants\":[\"Alice\",\"Bob\"]}",
        "refusal": null,
        "role": "assistant",
        "annotations": [],
        "audio": null,
        "function_call": null,
        "tool_calls": null,
        "parsed": {
          "name": "Science Fair",
          "date": "Friday",
          "participants": [
            "Alice",
            "Bob"
          ]
        }
      }
    }
  ],
  "created": 1789842333,
  "model": "gpt-5-nano-2025-08-07",
  "object": "chat.completion",
  "metadata": null,
  "moderation": null,
  "service_tier": "default",
  "system_fingerprint": null,
  "usage": {
    "completion_tokens": 1118,
    "prompt_tokens": 90,
    "total_tokens": 1208,
    "c

#### 4. `content` vs. `parsed`
The message holds the same answer twice:
- `content` is the JSON **text** the model wrote, a `str`.
- `parsed` is that text turned into a `CalendarEvent` **object** by the SDK.

If the model refuses, `parsed` is `None` and `refusal` holds its explanation.

In [6]:
message = completion.choices[0].message

print(f"message.content ({type(message.content).__name__}):")
print(message.content)
print()
print(f"message.parsed ({type(message.parsed).__name__}):")
print(repr(message.parsed))  # repr() keeps the class name, which print() alone drops for your own classes
print()
print("message.refusal:", message.refusal)

message.content (str):
{"name":"Science Fair","date":"Friday","participants":["Alice","Bob"]}

message.parsed (CalendarEvent):
CalendarEvent(name='Science Fair', date='Friday', participants=['Alice', 'Bob'])

message.refusal: None


#### 5. Using the fields
`event` is an ordinary Pydantic object: reach each field with a dot. Each value already has the Python type the class declares.

In [7]:
event = completion.choices[0].message.parsed

print("event.name:        ", event.name, f"({type(event.name).__name__})")
print("event.date:        ", event.date, f"({type(event.date).__name__})")
print("event.participants:", event.participants, f"({type(event.participants).__name__})")

event.name:         Science Fair (str)
event.date:         Friday (str)
event.participants: ['Alice', 'Bob'] (list)
